# 140 — Control clásico y control aprendido

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("robotics", seed=140)
assert result["kind"] == "robotics"
assert result["evidence"]
show(result)


## Solución 1 — Equilibrio solo-P

- (a) En equilibrio la acción compensa la fricción: `Kp·(10 − v) = 0.5·v` ⇒
  `v_∞ = 10·Kp/(Kp + 0.5)`.
- (b) Kp=1: 6.67; Kp=2: 8.0; Kp=5: 9.09; Kp=20: 9.76.
- (c) `10·Kp/(Kp+0.5) ≥ 9.9 ⇒ Kp ≥ 49.5`. No es razonable: una ganancia así
  amplifica ruido y, con Δt=0.1 finito, acerca el lazo discreto a la
  inestabilidad. La respuesta correcta de ingeniería es añadir el término I,
  no inflar Kp.


In [ ]:
for Kp in (1, 2, 5, 20, 49.5):
    v_inf = 10*Kp/(Kp+0.5)
    print(f"Kp={Kp}: v_inf={v_inf:.2f}, error={10-v_inf:.2f}")


## Solución 2 — P vs PI simulados

Solo-P con Kp=2 converge a v≈8.0 (error 2.0, coincide con la fórmula del
ejercicio 1) y no presenta sobreimpulso. El PI converge a v≈10.0 con un
sobreimpulso moderado (~5-10 %) mientras la integral descarga lo acumulado en
el arranque. La integral es quien aporta en régimen el `u=5` que la fricción
exige con error cero.


In [ ]:
def simula(Kp, Ki=0.0, pasos=300, dt=0.1, r=10.0):
    v, S, vmax = 0.0, 0.0, 0.0
    for _ in range(pasos):
        e = r - v
        S += e * dt
        u = Kp*e + Ki*S
        v = v + dt*(u - 0.5*v)
        vmax = max(vmax, v)
    return v, vmax

for nombre, args in (("P  Kp=2", (2.0, 0.0)), ("PI Kp=2 Ki=1", (2.0, 1.0))):
    vf, vmax = simula(*args)
    print(f"{nombre}: v_final={vf:.3f}, sobreimpulso={max(0, vmax-10):.3f}")


## Solución 3 — Diagnóstico

- (a) Oscilación sostenida: **Kp demasiado alto** (o retardo del lazo); bajar
  Kp o añadir D.
- (b) Error constante bajo carga: falta acción **integral** (perturbación
  constante); subir Ki o añadirlo.
- (c) Sobreimpulso solo en cambios grandes: **windup de la integral** durante
  el transitorio largo; anti-windup (saturar S o congelarla con error grande).
- (d) Picos de alta frecuencia: **Kd amplificando ruido** de medición; filtrar
  la derivada, derivar la medición en vez del error, o reducir Kd.


## Solución 4 — Diseño híbrido (respuesta modelo)

Una división defendible: control clásico (PID/impedancia) en los lazos
articulares de posición/fuerza — ejecuta a 1 kHz, es auditable y acota pares
máximos —, y política aprendida para la estrategia de inserción (búsqueda en
espiral, decidir reintentos, adaptarse a fricción variable), operando a
10-50 Hz sobre consignas. Riesgo del lado clásico: el modelo de contacto
lineal es falso cerca del encaje y puede clavar la pieza. Riesgo del lado
aprendido: estados fuera de distribución (pieza deformada) sin garantía de
comportamiento — mitigado porque sus salidas pasan por la envolvente de
fuerza del lazo clásico. El JSON del lab ilustra el contrato: evidencia
inspeccionable y limitaciones declaradas, que es lo que pedirías a ambas
mitades.


In [ ]:
from ai_evolution.labs import run_lab

result = run_lab("robotics", seed=140)
assert result["kind"] == "robotics" and result["evidence"]
print(result["limitations"])
